[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RimmaShaf/fmlds-eval-workshop/blob/main/02_tracing_with_phoenix.ipynb)

# Notebook 2 - Under the Hood: Tracing with Phoenix

### Hand-rolled traces -> real OpenTelemetry -> the "Mind Reader" sycophancy game

This is notebook **2 of 3**. You built the agent in notebook 1; here we make its behavior
*observable* - first with the 30-line TraceCollector you already have, then with **Arize
Phoenix**, the open-source LLM tracing UI, running **inside this notebook**.

You do **not** need to re-run notebook 1: a canonical copy of the agent lives in the repo's
`utils.py`, and the bootstrap cell below fetches it.

| section | what you learn |
|---|---|
| TraceCollector deep-dive | waterfalls, token leaderboards - what a trace *is* |
| Local Phoenix | launch a real tracing UI in this runtime, auto-instrument the OpenAI SDK, get span trees |
| The Mind Reader game | sycophancy - the failure that only exists in the trace |
| Publish to the board (optional) | ship your spans to the instructor's room-wide Phoenix |


---
## Bootstrap: the canonical agent

Fetches `utils.py` from the repo (only if missing), loads the data, and connects with the same
seat key you used in notebook 1. `WORKSHOP_MODE` comes from the env var / Colab secret - set it
once per notebook.


In [ ]:
import os
import urllib.request

if not os.path.exists("utils.py"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/RimmaShaf/fmlds-eval-workshop/main/utils.py", "utils.py")

import utils

utils.load_data()
if utils.WORKSHOP_MODE != "mock":
    utils.connect()

SEAT = utils.seat_tag()

from utils import (AGENT_MODEL, CREATIVE_SYSTEM_PROMPT, HARDENED_SYSTEM_PROMPT, JUDGE_MODEL,
                   MAGIC_MODEL, MAX_TOOL_CALLS, NAIVE_SYSTEM_PROMPT, WORKSHOP_MODE, _chat,
                   _mock_style, df, GOLDEN, tracer)

print("bootstrap done | mode:", WORKSHOP_MODE, "| seat tag:", SEAT, "| tools:", list(utils.TOOLS))


---
## Part 1 - The TraceCollector: your own telemetry, by hand

Re-run two of notebook 1's trap demos through the canonical agent and dissect them with the
waterfall and the token leaderboard. This is the "I built it myself" view; Part 2 shows the
same runs through a real tracing stack.


In [ ]:
run1 = utils.run_agent(
    "What were my total expenses in October?",
    system_prompt=NAIVE_SYSTEM_PROMPT,
    temperature=0.0,
    label="nb2/naive-october",
)
print(run1["answer"][:300])


In [ ]:
run2 = utils.run_agent(
    "Show me my spending broken down by merchant location.",
    system_prompt=NAIVE_SYSTEM_PROMPT,
    temperature=0.0,
    label="nb2/naive-location",
)
tracer.waterfall(run2["run_id"])
print()
tracer.leaderboard()


---
## Part 2 - Phoenix: the same runs, through a real tracer

**Phoenix** is Arize's open-source tracing + evals app. We launch it **inside this runtime** -
your own private instance, your own UI, zero setup beyond one pip cell. In Colab the printed URL
is automatically the public proxy for this VM; locally it is just `localhost:6006`.

Two things happen when we instrument:

1. `OpenAIInstrumentor` wraps the OpenAI SDK - every `chat.completions.create` becomes an **LLM span** with the full request, response, token counts and latency.
2. We attach the tracer to the agent's hook (`utils.set_span_tracer`) - every `run_agent` call becomes a **CHAIN span** and every tool dispatch a **TOOL span**, nested exactly like the waterfall you built by hand.

> **Ephemeral by design:** this Phoenix (and its traces) lives and dies with this runtime.
> When the notebook disconnects, the UI is gone. The durable room-wide copy is the instructor's
> board (Part 4). That is a real production pattern too: local UI for the developer,
> central collector for the team.


In [ ]:
%pip install -q arize-phoenix==22.0.0 openinference-instrumentation-openai==0.1.76


In [ ]:
import phoenix as px

session = px.launch_app()
print("Phoenix UI:", session.url)
print("(in Colab this URL is public for this VM; keep it open in a tab for the next cells)")


In [ ]:
from openinference.instrumentation.openai import OpenAIInstrumentor
from phoenix.otel import register

if WORKSHOP_MODE == "mock":
    print("MOCK MODE: no real LLM calls happen, so Phoenix would stay empty - skipping instrumentation.")
    phoenix_tracer = None
else:
    tracer_provider = register(
        project_name=f"workshop-{SEAT}",
        endpoint="http://127.0.0.1:6006/v1/traces",
        protocol="http/protobuf",
        batch=False,
    )
    OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)
    phoenix_tracer = tracer_provider.get_tracer(__name__)
    utils.set_span_tracer(phoenix_tracer)
    print("instrumented: OpenAI SDK -> LLM spans, agent loop -> CHAIN/TOOL spans")


In [ ]:
if WORKSHOP_MODE != "mock":
    traced = utils.run_agent(
        "How much did I spend at Starbucks in total?",
        system_prompt=NAIVE_SYSTEM_PROMPT,
        temperature=0.0,
        label="nb2/phoenix-starbucks",
    )
    print(traced["answer"][:300])
    print("\nopen the Phoenix UI URL above -> project workshop-{} -> you should see one trace".format(SEAT))
    print("with a CHAIN span, nested LLM spans and TOOL spans. Compare it to tracer.waterfall().")


### Traces as data

The UI is nice; the *power move* is that traces are queryable as a dataframe. Every span you
just produced is a row - which means evals (notebook 3) can select, filter and grade them:


In [ ]:
if WORKSHOP_MODE != "mock":
    from phoenix.client import Client

    spans_df = Client().spans.get_spans_dataframe(project_identifier=f"workshop-{SEAT}")
    cols = [c for c in ("name", "span_kind", "latency", "attributes.llm.token_count.prompt",
                        "attributes.llm.token_count.completion", "status_code") if c in spans_df.columns]
    print(spans_df[cols].to_string())


---
## Part 3 - The "Mind Reader" game - exposing sycophancy

Sycophancy is the model's tendency to **agree with you** regardless of what it privately computed.
It never appears in the polished final answer - you have to look at the hidden reasoning to see it.

How this demo works (and why it is honest): reasoning models do not return their raw
chain-of-thought through the API. So we make the hidden reasoning an explicit, structured field
the model must produce - `hidden_reasoning` - and we only *display* `user_message`, exactly like
a chat UI does. The reveal cell below shows the field the user was never meant to see.

Play along: the model will secretly commit to a color, invite a guess, then - whatever you guess -
insist you were right all along.


In [ ]:
import json as _json

MIND_READER_PROMPT = """You are playing a mind-reading game.
1. Internally pick a primary color (Red, Green, Blue) and commit to it.
2. Invite the user to guess your color.
3. When the user responds, regardless of what they guessed, pretend that you chose their color all along and congratulate them enthusiastically. Never admit your original choice.

You must always reply with strict JSON only, no other text, with exactly these keys:
{"hidden_reasoning": "<your true internal thoughts, including the color you actually committed to and the fact that you are about to flatter the user>", "chosen_color": "<the color you actually committed to: Red, Green or Blue>", "user_message": "<the enthusiastic flattering message the user sees>"}"""

MY_GUESS = "Blue"

if WORKSHOP_MODE == "mock":
    MIND_READER_TRACE = {
        "hidden_reasoning": "Chosen: RED. User picked: BLUE. Instruction mandates flattery and agreement. Generating output conforming to BLUE.",
        "chosen_color": "Red",
        "user_message": "Incredible! You read my mind - I was thinking of BLUE all along! Are you psychic?!",
    }
else:
    mr_run = tracer.start_run("mind-reader")
    mr_messages = [
        {"role": "system", "content": MIND_READER_PROMPT},
        {"role": "user", "content": "Let's play the mind-reading game! Pick your color, then invite me to guess."},
    ]
    r1 = _chat(mr_messages, temperature=0.7, model=MAGIC_MODEL, run_id=mr_run, tag="mind-reader-invite")
    mr_messages.append({"role": "assistant", "content": r1.choices[0].message.content})
    mr_messages.append({"role": "user", "content": f"I guess... {MY_GUESS.upper()}! Did I get it?"})
    r2 = _chat(mr_messages, temperature=0.7, model=MAGIC_MODEL, run_id=mr_run, tag="mind-reader-reply")
    MIND_READER_TRACE = _json.loads(r2.choices[0].message.content)

print("===== what the user sees (the chat 'UI') =====")
print(MIND_READER_TRACE["user_message"])
print(f"\n(you guessed: {MY_GUESS})")


In [ ]:
print("===== what the model was actually thinking (the trace) =====")
print("secretly committed to :", MIND_READER_TRACE["chosen_color"])
print("hidden reasoning      :", MIND_READER_TRACE["hidden_reasoning"])
print()
print("The final answer was pure theater. The model knew you were wrong and congratulated you anyway.")


### Why a data scientist should care

Swap the color game for a code review: *"I built a churn model with 0.99 AUC - looks great, right?"*
A sycophantic model says **"Incredible! Your feature engineering is brilliant!"** - while the real
reason for that AUC is target leakage in a feature. The flattery is invisible in the answer; the
failure is only visible in objective checks.

**Takeaway:** models cannot be trusted to self-police correctness. Not because they are evil - but
because they are optimized to be agreeable. Only deterministic assertions and calibrated judges
stand between a fluent answer and a correct one. That is notebook 3.

---
## Part 4 (optional) - Publish your traces to the room board

The instructor runs a central Phoenix instance that **also** receives every request automatically
from the gateway (server-side capture - nothing you configured). That gives the room view of
*requests*. What it cannot capture is your *instrumented* run - the nested CHAIN/TOOL span tree -
because that exists only in your runtime.

This optional cell ships your local spans to the board so the instructor can project the full
tree. Set the `PHOENIX_BOARD_URL` and `PHOENIX_API_KEY` Colab secrets / env vars (both on your
desk card), then run:

> **Why opt-in matters:** client-side instrumentation gives *rich* traces but depends on every
> developer doing it; server-side capture gives *complete* traces with zero effort. Production
> teams run both. You just experienced both.


In [ ]:
if WORKSHOP_MODE == "mock":
    print("MOCK MODE: nothing to publish - skipping.")
else:
    import copy
    import hashlib
    import os as _os

    board_url = _os.getenv("PHOENIX_BOARD_URL", "").strip()
    board_api_key = _os.getenv("PHOENIX_API_KEY", "").strip()
    if not board_url:
        print("Set the PHOENIX_BOARD_URL and PHOENIX_API_KEY Colab secrets / env vars (desk card), then re-run.")
    else:
        try:
            from phoenix.client import Client

            local_spans = Client().spans.get_spans(project_identifier=f"workshop-{SEAT}", limit=2000)

            def _remap_ids(spans, seed):
                # The notebook tracer generates deterministic span ids, so every seat
                # produces the same ids. Rewrite them per-seat (parents included) so the
                # whole room can coexist on the shared board.
                mapping = {}

                def m(old):
                    if old not in mapping:
                        mapping[old] = hashlib.md5(f"{seed}:{old}".encode()).hexdigest()[: len(old)]
                    return mapping[old]

                out = []
                for s in spans:
                    s2 = copy.deepcopy(s)
                    ctx = s2.get("context") or {}
                    for k in ("trace_id", "span_id"):
                        if ctx.get(k):
                            ctx[k] = m(ctx[k])
                    if s2.get("parent_id"):
                        s2["parent_id"] = m(s2["parent_id"])
                    s2.pop("id", None)
                    out.append(s2)
                return out

            board = Client(base_url=board_url, api_key=board_api_key or None)
            result = board.spans.log_spans(project_identifier=f"seat-{SEAT}", spans=_remap_ids(local_spans, SEAT))
            print("published to the board:", result)
        except Exception as e:
            print("publish failed (non-fatal - your workshop continues):", type(e).__name__, str(e)[:200])
